# VAE Denoising Model Training

Train a Variational Autoencoder (VAE) to denoise gravitational wave signals using curriculum learning.
The SNR (Signal-to-Noise Ratio) starts high (clean signals) and linearly decreases to low SNR (noisy signals) over epochs.

## 1. Import Required Libraries

In [1]:
import sys
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Add src to path
sys.path.insert(0, str(Path.cwd() / "src"))

from starccato_flow.training import VAEDenoisingTrainer
from starccato_flow.utils.defaults import DEVICE

print(f"Device: {DEVICE}")
print(f"Working directory: {Path.cwd()}")

MPS device found
Device: mps
Working directory: /Users/tarineccleston/Desktop/starccato/starccato-flow


/Users/tarineccleston/Desktop/starccato/starccato-flow/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Training Configuration

Configure the VAE denoising trainer with curriculum learning parameters.

In [2]:
# Training hyperparameters
config = {
    "num_epochs": 256,
    "toy": False,  # Use full CCSN dataset
    "batch_size": 32,
    "learning_rate": 1e-3,
    "beta": 1.0,  # VAE KL weight: 0=autoencoder, 1=full VAE
    "curriculum_snr_start": 200.0,  # Start with clean signals
    "curriculum_snr_end": 8.0,  # End with noisy signals
    "gradient_clip": 5.0,
    "checkpoint_interval": 16,  # Save checkpoint every N epochs
    "z_dim": 64,  # Latent dimension
}

print("Training Configuration:")
for key, value in config.items():
    print(f"  {key}: {value}")

Training Configuration:
  num_epochs: 256
  toy: False
  batch_size: 32
  learning_rate: 0.001
  beta: 1.0
  curriculum_snr_start: 200.0
  curriculum_snr_end: 8.0
  gradient_clip: 5.0
  checkpoint_interval: 16
  z_dim: 64


## 3. Initialize Trainer

In [3]:
# Initialize the VAE denoising trainer
trainer = VAEDenoisingTrainer(
    num_epochs=config["num_epochs"],
    toy=config["toy"],
    batch_size=config["batch_size"],
    learning_rate=config["learning_rate"],
    beta=config["beta"],
    curriculum_snr_start=config["curriculum_snr_start"],
    curriculum_snr_end=config["curriculum_snr_end"],
    gradient_clip=config["gradient_clip"],
    checkpoint_interval=config["checkpoint_interval"],
    z_dim=config["z_dim"],
)

print(f"Trainer initialized")
print(f"Model: {trainer.model}")
print(f"Training dataset size: {len(trainer.training_dataset)}")
print(f"Validation dataset size: {len(trainer.validation_dataset)}")

TypeError: __init__() got an unexpected keyword argument 'gradient_clip'

## 4. Train the Model

Train the VAE with curriculum learning (SNR decreases from 200 to 8 over epochs).

In [ ]:
# Train the model
print("Starting training...")
trainer.train()

## 5. Visualize Training Losses

Plot training and validation losses over epochs.

In [ ]:
# Load and plot losses
losses_file = Path("outdir") / "vae_denoising_losses.npz"
losses_data = np.load(losses_file)

train_loss = losses_data["train_loss"]
train_recon = losses_data["train_recon"]
train_kl = losses_data["train_kl"]
val_loss = losses_data["val_loss"]
val_recon = losses_data["val_recon"]
val_kl = losses_data["val_kl"]

epochs = np.arange(len(train_loss))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Total loss
axes[0].plot(epochs, train_loss, label="Train", linewidth=2)
axes[0].plot(epochs, val_loss, label="Val", linewidth=2)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Total Loss")
axes[0].set_title("ELBO Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Reconstruction loss
axes[1].plot(epochs, train_recon, label="Train", linewidth=2)
axes[1].plot(epochs, val_recon, label="Val", linewidth=2)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("MSE Loss")
axes[1].set_title("Reconstruction Loss")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# KL divergence
axes[2].plot(epochs, train_kl, label="Train", linewidth=2)
axes[2].plot(epochs, val_kl, label="Val", linewidth=2)
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("KL Divergence")
axes[2].set_title("KL Divergence")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("plots/vae_denoising_losses.png", dpi=300, bbox_inches="tight")
plt.show()

print(f"Final train loss: {train_loss[-1]:.6f}")
print(f"Final val loss: {val_loss[-1]:.6f}")
print(f"Best val loss: {val_loss.min():.6f} at epoch {val_loss.argmin()}")

## 6. Visualize Curriculum Learning Schedule

Show how SNR (noise level) changes over training epochs.

In [ ]:
# Compute curriculum schedule
snr_start = config["curriculum_snr_start"]
snr_end = config["curriculum_snr_end"]
num_epochs = config["num_epochs"]

curriculum_snr = np.array([
    snr_start + (epoch / max(num_epochs - 1, 1)) * (snr_end - snr_start)
    for epoch in range(num_epochs)
])

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(curriculum_snr, linewidth=2.5, color="C0")
ax.fill_between(range(len(curriculum_snr)), curriculum_snr, alpha=0.3, color="C0")
ax.set_xlabel("Epoch", fontsize=12)
ax.set_ylabel("SNR (dB)", fontsize=12)
ax.set_title("Curriculum Learning Schedule: SNR Over Training", fontsize=14)
ax.grid(True, alpha=0.3)
ax.invert_yaxis()  # Invert so cleaner signals (higher SNR) appear at top

plt.tight_layout()
plt.savefig("plots/vae_denoising_curriculum.png", dpi=300, bbox_inches="tight")
plt.show()

print(f"SNR starts at: {snr_start:.1f} dB (clean signals)")
print(f"SNR ends at: {snr_end:.1f} dB (noisy signals)")

## 7. Generate Denoised Examples

Reconstruct validation signals using the trained VAE.

In [ ]:
# Get validation signals and reconstruct with trained model
import torch

# Set dataset to final SNR (end of curriculum)
trainer.validation_dataset.update_snr(config["curriculum_snr_end"])

# Get sample batch
dataloader = trainer.val_dataloader
batch = next(iter(dataloader))

# Extract noisy signals (batch, channels, time)
noisy_signals = batch["noisy_signals"].to(trainer.device)
clean_signals = batch["clean_signals"].to(trainer.device)

# Reconstruct with trained model
with torch.no_grad():
    mean, log_var = trainer.model.encode(noisy_signals)
    z = trainer.model.reparameterize(mean, log_var)
    reconstructed = trainer.model.decode(z)

# Move to CPU and numpy
noisy = noisy_signals.cpu().numpy()
clean = clean_signals.cpu().numpy()
recon = reconstructed.cpu().numpy()

# Plot first 8 examples
n_samples = min(8, noisy.shape[0])
fig, axes = plt.subplots(n_samples, 3, figsize=(12, 2*n_samples))

for i in range(n_samples):
    # Noisy
    axes[i, 0].plot(noisy[i, 0], linewidth=0.8, color="C3", alpha=0.7)
    axes[i, 0].set_ylabel(f"Sample {i}", fontsize=10)
    if i == 0:
        axes[i, 0].set_title("Noisy Signal", fontsize=11)
    axes[i, 0].grid(True, alpha=0.2)
    
    # Reconstructed
    axes[i, 1].plot(recon[i, 0], linewidth=0.8, color="C2")
    if i == 0:
        axes[i, 1].set_title("Reconstructed (Denoised)", fontsize=11)
    axes[i, 1].grid(True, alpha=0.2)
    
    # Clean
    axes[i, 2].plot(clean[i, 0], linewidth=0.8, color="C0", alpha=0.7)
    if i == 0:
        axes[i, 2].set_title("Clean Signal", fontsize=11)
    axes[i, 2].grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig("plots/vae_denoising_examples.png", dpi=300, bbox_inches="tight")
plt.show()

print(f"Displayed {n_samples} validation examples")

## 8. Training Summary and Model Information

In [ ]:
# Print training summary
print("="*60)
print("VAE DENOISING TRAINING SUMMARY")
print("="*60)
print(f"\nTraining Configuration:")
print(f"  Epochs: {config['num_epochs']}")
print(f"  Batch size: {config['batch_size']}")
print(f"  Learning rate: {config['learning_rate']}")
print(f"  Beta (KL weight): {config['beta']}")
print(f"  Latent dimension: {config['z_dim']}")

print(f"\nCurriculum Learning:")
print(f"  Initial SNR: {config['curriculum_snr_start']:.1f} dB (clean)")
print(f"  Final SNR: {config['curriculum_snr_end']:.1f} dB (noisy)")

print(f"\nLoss Metrics:")
print(f"  Final training loss: {train_loss[-1]:.6f}")
print(f"  Final validation loss: {val_loss[-1]:.6f}")
print(f"  Best validation loss: {val_loss.min():.6f} (epoch {val_loss.argmin()})")

print(f"\nModel Checkpoints:")
print(f"  Saved every {config['checkpoint_interval']} epochs")
print(f"  Final model weights: outdir/vae_denoising_weights_final.pt")

print(f"\nOutput Files:")
print(f"  Loss data: outdir/vae_denoising_losses.npz")
print(f"  Loss plot: plots/vae_denoising_losses.png")
print(f"  Curriculum plot: plots/vae_denoising_curriculum.png")
print(f"  Denoising examples: plots/vae_denoising_examples.png")
print("="*60)